# STEP 6 - ELASTIC SEARCH AND INVERTED INDEXES

In this notebook we will:

1. Connect to Elasticsearch running in Docker
2. Verify the connection
3. Create an explicit index mapping
4. Load our document chunks
5. Bulk-index the chunks
6. Run basic match queries
7. Inspect BM25 scores
8. Run exact-match queries
9. Experiment with analyzers
10. Compare lexical search with vector search

                     DOCUMENT
                         │
                         ▼
                   Split into chunks
                         │
              ┌──────────┴──────────┐
              │                     │
              ▼                     ▼
        Generate embedding      Elasticsearch
              │                     │
              ▼                     ▼
        Vector database       Inverted index
              │                     │
              │                     │
              └──────────┬──────────┘
                         │
                         ▼
                    USER QUERY
                         │
              ┌──────────┴──────────┐
              │                     │
              ▼                     ▼
        Vector retrieval      Keyword retrieval
              │                     │
              └──────────┬──────────┘
                         ▼
                    Combine results
                         │
                         ▼
                      Rerank
                         │
                         ▼
                         LLM
                         │
                         ▼
                       Answer

In [33]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

try:
    response = es.info()
    print(response)

except Exception as e:
    print("TYPE:", type(e).__name__)
    print("ERROR:")
    print(e)

    if hasattr(e, "body"):
        print("\nBODY:")
        print(e.body)

{'name': '0e1f447b24d9', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'lHCCr2OyQ0yMLKjGmwdmBg', 'version': {'number': '8.19.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '93788a8c2882eb5b606510680fac214cff1c7a22', 'build_date': '2025-07-23T22:10:18.138212839Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [34]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".." ).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## helper method to reload specific file
import importlib
import config
importlib.reload(config)

<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

In [35]:
from pathlib import Path
import numpy as np
import gradio as gr
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL, RETRIEVAL_MODE_BM25, RETRIEVAL_MODE_VECTOR
import chromadb

# 1. Initialize client & resolve paths
client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma= chromadb.HttpClient(
    host="localhost",
    port=8000,
)

# vector or bm25
RETRIEVAL_MODE= 'vector' 

In [36]:
# 2. Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]

In [37]:
INDEX_NAME = "rag_documents"

In [38]:
es.indices.exists(index=INDEX_NAME)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

print("Index reset complete.")

Index reset complete.


In [39]:
mapping = {
    "mappings": {
        "properties": {
            "chunk_id": {
                "type": "keyword"
            },
            "text": {
                "type": "text"
            }
        }
    }
}

In [40]:
response = es.indices.create(
    index=INDEX_NAME,
    body=mapping
)

print(response)

{'acknowledged': True, 'shards_acknowledged': True, 'index': 'rag_documents'}


In [41]:
print(es.indices.get_mapping(index=INDEX_NAME))

{'rag_documents': {'mappings': {'properties': {'chunk_id': {'type': 'keyword'}, 'text': {'type': 'text'}}}}}


In [42]:
documents[:2]

[{'id': 0,
  'text': 'Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service'},
 {'id': 1,
  'text': "John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life."}]

In [43]:
print("Number of chunks:", len(documents))

Number of chunks: 22


In [44]:
actions = []

for doc in documents:
    actions.append({
        "_index": INDEX_NAME,
        "_id": str(doc["id"]),
        "_source": {
            "chunk_id": str(doc["id"]),
            "text": doc["text"]
        }
    })

In [45]:
actions[0]

{'_index': 'rag_documents',
 '_id': '0',
 '_source': {'chunk_id': '0',
  'text': 'Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service'}}

In [46]:
from elasticsearch.helpers import bulk
success, failed = bulk(
    es,
    actions
)

print("Successfully indexed:", success)
print("Failed:", failed)

Successfully indexed: 22
Failed: []


In [47]:
count_result = es.count(
    index=INDEX_NAME
)

print(count_result)

{'count': 22, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}}


In [48]:
result = es.get(
    index=INDEX_NAME,
    id="0"
)

print(result)

{'_index': 'rag_documents', '_id': '0', '_version': 1, '_seq_no': 0, '_primary_term': 1, 'found': True, '_source': {'chunk_id': '0', 'text': 'Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service'}}


In [49]:
query = {
    "query": {
        "match": {
            "text": "John innovation"
        }
    }
}

results = es.search(
    index=INDEX_NAME,
    body=query
)

In [50]:
for hit in results["hits"]["hits"]:
    print("Score:", hit["_score"])
    print("Chunk ID:", hit["_source"]["chunk_id"])
    print("Text:", hit["_source"]["text"])
    print("-" * 80)

Score: 3.2879915
Chunk ID: 0
Text: Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service
--------------------------------------------------------------------------------
Score: 2.4184444
Chunk ID: 21
Text: John's fictional story illustrates how curiosity, continuous learning, compassion, and perseverance can shape a meaningful career. Although entirely fictional, his biography reflects values shared by many successful professionals: a commitment to lifelong education, ethical innovation, collaborative leadership, and service to the broader community.
--------------------------------------------------------------------------------
Score: 0.0910714
Chunk ID: 18
Text: John also became increasingly interested in lifelong learning. He believed that technology professionals should continuously adapt as industries evolved. He frequently reminded younger engineers that communication, empathy, and critical thinking would remain valuable regardless of technologic

In [51]:
query = {
    "size": 3,
    "query": {
        "match": {
            "text": "John innovation"
        }
    }
}

In [52]:
results = es.search(
    index=INDEX_NAME,
    body=query
)

In [53]:
def elasticsearch_search(query_text, top_k=3):
    
    query = {
        "size": top_k,
        "query": {
            "match": {
                "text": query_text
            }
        }
    }

    results = es.search(
        index=INDEX_NAME,
        body=query
    )

    return results

In [54]:
results = elasticsearch_search(
    "John innovation"
)

for hit in results["hits"]["hits"]:
    print("Score:", hit["_score"])
    print(hit["_source"]["text"])
    print()

Score: 3.2879915
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service

Score: 2.4184444
John's fictional story illustrates how curiosity, continuous learning, compassion, and perseverance can shape a meaningful career. Although entirely fictional, his biography reflects values shared by many successful professionals: a commitment to lifelong education, ethical innovation, collaborative leadership, and service to the broader community.

Score: 0.0910714
John also became increasingly interested in lifelong learning. He believed that technology professionals should continuously adapt as industries evolved. He frequently reminded younger engineers that communication, empathy, and critical thinking would remain valuable regardless of technological advances. According to John, successful careers were built not only on technical skills but also on integrity, resilience, and collaboration.



In [55]:
query = {
    "query": {
        "term": {
            "chunk_id": "0"
        }
    }
}

results = es.search(
    index=INDEX_NAME,
    body=query
)

for hit in results["hits"]["hits"]:
    print(hit["_source"])

{'chunk_id': '0', 'text': 'Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service'}


In [56]:
match_query = {
    "query": {
        "match": {
            "text": "John Doe"
        }
    }
}

term_query = {
    "query": {
        "term": {
            "chunk_id": "0"
        }
    }
}

In [57]:
analysis_result = es.indices.analyze(
    body={
        "text": "The employees were working on connected systems."
    }
)

analysis_result

ObjectApiResponse({'tokens': [{'token': 'the', 'start_offset': 0, 'end_offset': 3, 'type': '<ALPHANUM>', 'position': 0}, {'token': 'employees', 'start_offset': 4, 'end_offset': 13, 'type': '<ALPHANUM>', 'position': 1}, {'token': 'were', 'start_offset': 14, 'end_offset': 18, 'type': '<ALPHANUM>', 'position': 2}, {'token': 'working', 'start_offset': 19, 'end_offset': 26, 'type': '<ALPHANUM>', 'position': 3}, {'token': 'on', 'start_offset': 27, 'end_offset': 29, 'type': '<ALPHANUM>', 'position': 4}, {'token': 'connected', 'start_offset': 30, 'end_offset': 39, 'type': '<ALPHANUM>', 'position': 5}, {'token': 'systems', 'start_offset': 40, 'end_offset': 47, 'type': '<ALPHANUM>', 'position': 6}]})

In [58]:
for token in analysis_result["tokens"]:
    print(token["token"])

the
employees
were
working
on
connected
systems


In [59]:
analysis_result = es.indices.analyze(
    body={
        "text": "John JOHN John"
    }
)

for token in analysis_result["tokens"]:
    print(token["token"])

john
john
john


In [60]:
analysis_settings = {
    "settings": {
        "analysis": {
            "analyzer": {
                "my_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": [
                        "lowercase"
                    ]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "text": {
                "type": "text",
                "analyzer": "my_analyzer"
            },
            "chunk_id": {
                "type": "keyword"
            }
        }
    }
}